In [1]:
# Import Libraries
import os
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import ollama

[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.4.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'nn' is not defined

In [ ]:
# Extract Text
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

In [ ]:
# Chunking with Overlap
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start &lt; len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

pdf_filename = "coffee_maker.pdf"
raw_text = extract_text_from_pdf(pdf_filename)
chunks = chunk_text(raw_text, chunk_size=500, overlap=50)
print(f"--- Step 1 &amp; 2: PDF Chunking Complete ---")
print(f"Total Chunks Created: {len(chunks)}\n")

for idx, chunk in enumerate(chunks):
    print(f"--- CHUNK {idx + 1} ---")
    print(chunk.strip())
    print("-" * 40)

In [ ]:
# Test Questions
questions = [
    "How often should I descale the coffee maker and what solution should I use?",
    "How long does the warming plate keep the coffee hot after brewing completes?",
    "What is the first step to set up the coffee maker?",
    "What should I do if the screen stops responding or freezes?" # Rephrased question
]

In [ ]:
# TF-IDF Retrieval 
tfidf_vectorizer = TfidfVectorizer()
tfidf_docs_matrix = tfidf_vectorizer.fit_transform(chunks)

# Embedding Retrieval 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedding_model.encode(chunks, convert_to_tensor=True)

In [ ]:
# Retrieval Comparison
print("\n" + "="*50)
print("--- RETRIEVAL COMPARISON (TF-IDF vs EMBEDDINGS) ---")
print("="*50)
best_chunks_for_rag = []

for q_idx, q in enumerate(questions, 1):
    print(f"\n[Question {q_idx}]: \"{q}\"")
    
    # TF-IDF
    q_tfidf_vec = tfidf_vectorizer.transform([q])
    tfidf_scores = cosine_similarity(q_tfidf_vec, tfidf_docs_matrix)
    best_tfidf_idx = tfidf_scores.argmax()
    
    # Embedding
    q_embed_vec = embedding_model.encode(q, convert_to_tensor=True)
    embed_scores = util.cos_sim(q_embed_vec, doc_embeddings)
    best_embed_idx = embed_scores.argmax().item()
    
    print(f"  • TF-IDF Top Match   : Chunk #{best_tfidf_idx + 1} (Score: {tfidf_scores[best_tfidf_idx]:.4f})")
    print(f"  • Embedding Top Match: Chunk #{best_embed_idx + 1} (Score: {embed_scores[best_embed_idx].item():.4f})")
    
    if best_tfidf_idx == best_embed_idx:
        print("  -&gt; Result: BOTH methods retrieved the same chunk.")
    else:
        print("  -&gt; Result: DIFFERENT chunks retrieved.")
        
    best_chunks_for_rag.append((q, chunks[best_embed_idx]))

In [ ]:
# Local LLM Generation
print("\n" + "="*50)
print("--- GENERATING GROUNDED ANSWERS WITH OLLAMA ---")
print("="*50)

for q_idx, (q, chunk_context) in enumerate(best_chunks_for_rag, 1):
    prompt = f"""You are an internal appliance support assistant. Answer the user's question using ONLY the information in the context below. If the context doesn't contain the answer, say you don't have that information.

Context:
{chunk_context}
Question:
{q}
Answer:"""
    try:
        response = ollama.generate(model="llama3.2", prompt=prompt)
        print(f"\n[Q{q_idx} Query]: {q}")
        print(f"[Generated Response]:\n{response['response'].strip()}")
        print("-" * 50)
    except Exception as e:
        print(f"Could not reach Ollama: {e}")